In [8]:
import pandas as pd
import torch
from pathlib import Path
print("GPU available:", torch.cuda.is_available())

from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset
from datasets import Dataset

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering, Trainer, TrainingArguments, AutoModelForSeq2SeqLM

from evaluate import load as load_metric

import numpy as np

import re
import string


GPU available: True


In [9]:
dataset = load_dataset("coastalcph/tydi_xor_rc")


df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

# Selected languages
langlst = ['ko','ar','te']
lang_map = {
    "ar": "arb_Arab",
    "ko": "kor_Hang",
    "te": "tel_Telu"
}

# Training and Validation data for selected languages
df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]
 
#Training and Validation data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]


saved_model_exists = all(
    Path("english-qa", filename).is_file()
    for filename in ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"]
)
model_source = "english-qa" if saved_model_exists else "bert-base-cased"

qa_tokenizer = AutoTokenizer.from_pretrained(
    model_source, use_fast=True, local_files_only=saved_model_exists
)




In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


nllb_name = "facebook/nllb-200-distilled-600M"

nllb_tokenizer = AutoTokenizer.from_pretrained(nllb_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
nllb_model = nllb_model.to(device)
nllb_model.eval()






def translate(question, lang):
	source_lang = lang_map[lang]
	nllb_tokenizer.src_lang = source_lang

	inputs = nllb_tokenizer(
		question,
		return_tensors="pt",
		truncation=True,
		max_length=512
	).to(device)

	with torch.inference_mode():
		translated_tokens = nllb_model.generate(
			**inputs,
			forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("eng_Latn"),
		)

	return nllb_tokenizer.batch_decode(
		translated_tokens,
		skip_special_tokens=True
	)[0]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [11]:
from pathlib import Path
import pandas as pd

from tqdm.auto import tqdm
tqdm.pandas()


save_path = Path("translated_dataset.parquet")

if save_path.exists():
    print("Loading previously translated dataset...")
    df_train_filtered = pd.read_parquet(save_path)

else:
    print("Translating questions...")

    df_train_filtered["translated_question"] = df_train_filtered.progress_apply(
        lambda row: translate(
            row["question"],
            row["lang"]
        ),
        axis=1
    )

    df_train_filtered.to_parquet(save_path, index=False)

Loading previously translated dataset...


In [12]:
df_train_filtered

,question,context,lang,answerable,answer_start,answer,answer_inlang,translated_question
0,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,Who is the winner of the Thirty Years' War?
1,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,Who discovered the X-ray?
2,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,When was the last time the Olympics were held ...
3,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,What's the oldest broadcaster in the world?
4,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,Where is the Palestinian capital?
...,...,...,...,...,...,...,...,...
6330,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,NaN,When did Somalia make its second inauguration?
6331,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,NaN,What was the first transportation system in th...
6332,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,NaN,Who is the leader of Egypt in 2019?
6333,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,NaN,What is the most densely populated city in Ger...


In [ ]:
torch.manual_seed(42)

def prepare_examples(examples):

	tokens = qa_tokenizer(
		examples["translated_question"],
		examples["context"],
		truncation="only_second",
		max_length=384,
		stride=128,
		return_overflowing_tokens=True,
		return_offsets_mapping=True,
		padding="max_length",
	)
	example_numbers = tokens.pop("overflow_to_sample_mapping")
	offsets = tokens.pop("offset_mapping")
	start_positions = []
	end_positions = []

	for i, example_number in enumerate(example_numbers):
		cls_position = tokens["input_ids"][i].index(qa_tokenizer.cls_token_id)
		start_position = cls_position
		end_position = cls_position

		if examples["answerable"][example_number]:
			answer_start = int(examples["answer_start"][example_number])
			answer_end = answer_start + len(examples["answer"][example_number])
			context_tokens = [
				j for j, part in enumerate(tokens.sequence_ids(i)) if part == 1
			]
			if (context_tokens
				and offsets[i][context_tokens[0]][0] <= answer_start
				and offsets[i][context_tokens[-1]][1] >= answer_end):
				answer_tokens = [
					j for j in context_tokens
					if offsets[i][j][0] < answer_end
					and offsets[i][j][1] > answer_start
				]
				if answer_tokens:
					start_position = answer_tokens[0]
					end_position = answer_tokens[-1]

		start_positions.append(start_position)
		end_positions.append(end_position)

	tokens["start_positions"] = start_positions
	tokens["end_positions"] = end_positions
	return tokens

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    model_source,
    local_files_only=saved_model_exists
)

if saved_model_exists:
	training_args = TrainingArguments(output_dir="english-qa", report_to="none")
	trainer = Trainer(model=qa_model, args=training_args)
	print("Loaded saved model. Skipping training.")
else:
	train_examples = Dataset.from_pandas(df_train_filtered, preserve_index=False)
	train_features = train_examples.map(
		prepare_examples, batched=True,
		remove_columns=train_examples.column_names,
	)

	training_args = TrainingArguments(
		output_dir="english-qa",
		num_train_epochs=3,
		per_device_train_batch_size=8,
		# per_device_train_batch_size=2,
		# gradient_accumulation_steps=4,
		fp16=True,
		learning_rate=3e-5,
		seed=42, 
		save_strategy="no", 
		report_to="none",
		full_determinism=True,
	)

	trainer = Trainer(
		model=qa_model,
		args=training_args,
		train_dataset=train_features
	)

	trainer.train()

	trainer.save_model("english-qa")
	qa_tokenizer.save_pretrained("english-qa")
	print("Saved final model and tokenizer.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded saved model. Skipping training.


In [14]:

save_path = Path("translated_validation_dataset.parquet")

if save_path.exists():
    print("Loading previously translated dataset...")
    df_validation_filtered = pd.read_parquet(save_path)

else:
    print("Translating questions...")

    df_validation_filtered["translated_question"] = df_validation_filtered.progress_apply(
        lambda row: translate(
            row["question"],
            row["lang"]
        ),
        axis=1
    )

    df_validation_filtered.to_parquet(save_path, index=False)


validation_examples = Dataset.from_pandas(
    df_validation_filtered, preserve_index=False
)



def prepare_validation(examples):
    tokens = qa_tokenizer(
        examples["translated_question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Local example indices within this preprocessing batch.
    example_numbers = tokens.pop("overflow_to_sample_mapping")
    tokens["example_index"] = example_numbers

    for i in range(len(tokens["input_ids"])):
        sequence_ids = tokens.sequence_ids(i)
        # Only context tokens can belong to an extracted answer.
        tokens["offset_mapping"][i] = [
            offset if sequence_ids[j] == 1 else None
            for j, offset in enumerate(tokens["offset_mapping"][i])
        ]

    return tokens

# One batch makes example_index refer to the whole validation dataset.
validation_features = validation_examples.map(
    prepare_validation,
    batched=True,
    batch_size=len(validation_examples),
    remove_columns=validation_examples.column_names,
)

# Metadata is needed for decoding, but must not be passed to the model.
model_inputs = validation_features.remove_columns(
    ["offset_mapping", "example_index"]
)
start_logits, end_logits = trainer.predict(model_inputs).predictions

n = len(validation_examples)
null_scores = np.full(n, np.inf)
span_scores = np.full(n, -np.inf)
best_answers = [""] * n

# A fixed decoding choice: number of WordPiece tokens, not words.
MAX_ANSWER_LENGTH = 30

for feature_index, feature in enumerate(validation_features):
    example_index = feature["example_index"]
    offsets = feature["offset_mapping"]
    starts = start_logits[feature_index]
    ends = end_logits[feature_index]

    cls_index = feature["input_ids"].index(qa_tokenizer.cls_token_id)
    null_scores[example_index] = min(
        null_scores[example_index],
        float(starts[cls_index] + ends[cls_index]),
    )

    # Search all valid context spans up to MAX_ANSWER_LENGTH.
    for start, start_offset in enumerate(offsets):
        if start_offset is None or start_offset[1] <= start_offset[0]:
            continue

        for end in range(
            start, min(start + MAX_ANSWER_LENGTH, len(offsets))
        ):
            end_offset = offsets[end]
            if end_offset is None:
                break
            if end_offset[1] <= end_offset[0]:
                continue

            score = float(starts[start] + ends[end])
            if score > span_scores[example_index]:
                span_scores[example_index] = score
                context = validation_examples[example_index]["context"]
                best_answers[example_index] = context[
                    start_offset[0]:end_offset[1]
                ]

assert np.isfinite(null_scores).all()
assert np.isfinite(span_scores).all()

no_answer_scores = null_scores - span_scores

Loading previously translated dataset...


Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

In [15]:
scores = np.asarray(no_answer_scores)
gold_answerable = df_validation_filtered["answerable"].to_numpy(dtype=bool)

thresholds = []

# First, test a cutoff that predicts no answer for everyone.
threshold_below_all_scores = scores.min() - 1
thresholds.append(threshold_below_all_scores)

# np.unique returns the distinct scores in ascending order.
unique_scores = np.unique(scores)

# Then test each score as a cutoff.
for score in unique_scores:
    thresholds.append(score)

best_threshold = thresholds[0]
best_macro_f1 = -1

for threshold in thresholds:
    predicted_answerable = scores <= threshold

    macro_f1 = f1_score(
        gold_answerable,
        predicted_answerable,
        labels=[False, True],
        average="macro",
        zero_division=0,
    )

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_threshold = threshold

NO_ANSWER_THRESHOLD = best_threshold

print("Threshold:", NO_ANSWER_THRESHOLD)
print("Validation macro-F1:", best_macro_f1)

Threshold: -2.905228614807129
Validation macro-F1: 0.8658312447786132


In [16]:
predicted_answers = [
    answer if score <= NO_ANSWER_THRESHOLD else ""
    for answer, score in zip(best_answers, scores)
]

In [17]:


# Lab 6 recommends squad_v2 for the project's unanswerable examples.
squad_metric = load_metric("squad_v2")

evaluation = df_validation_filtered.copy()
evaluation["prediction"] = predicted_answers
evaluation["pred_answerable"] = evaluation["prediction"] != ""

results = []
for language in ["overall", "ar", "ko", "te"]:
    group = evaluation if language == "overall" else evaluation[evaluation["lang"] == language]
    formatted_predictions = []
    gold = []

    for example_id, example in group.iterrows():
        formatted_predictions.append({
            "id": str(example_id),
            "prediction_text": example["prediction"],
            # Required field: encode the already-made abstention decision.
            # This is a binary indicator, not a calibrated probability.
            "no_answer_probability": float(not example["pred_answerable"]),
        })
        gold.append({
            "id": str(example_id),
            "answers": {
                "text": [example["answer"]] if example["answerable"] else [],
                "answer_start": [int(example["answer_start"])] if example["answerable"] else [],
            },
        })

    squad_results = squad_metric.compute(
        references=gold, predictions=formatted_predictions
    )
    report = classification_report(
        group["answerable"], group["pred_answerable"],
        labels=[False, True], target_names=["unanswerable", "answerable"],
        output_dict=True, zero_division=0,
    )

    row = {
        "language": language,
        "N": len(group),
        "SQuAD_v2_EM": squad_results["exact"],
        "SQuAD_v2_token_F1": squad_results["f1"],
        "answerability_macro_F1": 100 * report["macro avg"]["f1-score"],
    }
    for label in ["unanswerable", "answerable"]:
        row[f"{label}_P"] = 100 * report[label]["precision"]
        row[f"{label}_R"] = 100 * report[label]["recall"]
        row[f"{label}_F1"] = 100 * report[label]["f1-score"]
    results.append(row)

results_table = pd.DataFrame(results)
display(results_table.round(2))

,language,N,SQuAD_v2_EM,SQuAD_v2_token_F1,answerability_macro_F1,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1
0,overall,1155,57.66,67.15,86.58,80.13,73.78,76.83,95.72,96.97,96.34
1,ar,415,60.48,69.87,93.71,84.48,94.23,89.09,99.16,97.52,98.33
2,ko,356,57.58,69.85,88.86,70.83,89.47,79.07,99.40,97.92,98.65
3,te,384,54.69,61.70,79.66,79.71,59.14,67.90,87.94,95.19,91.42


In [18]:
def normalize_answer(text):
    text = text.lower()
    text = "".join(c for c in text if c not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

examples = evaluation.copy()
examples["gold_answer"] = examples["answer"].where(
    examples["answerable"], ""
)

errors = examples[
    examples["prediction"].map(normalize_answer)
    != examples["gold_answer"].map(normalize_answer)
]

selected_errors = errors.groupby("lang", sort=False).head(4)

print("Selected errors:", len(selected_errors))
print(selected_errors["lang"].value_counts())

for index, row in selected_errors.iterrows():
    print(f"\nExample {index} ({row['lang']})")
    print("Question:", row["question"])
    print("Context:", row["context"])
    print("Gold answer:", repr(row["gold_answer"]))
    print("Prediction:", repr(row["prediction"]))

Selected errors: 12
lang
te    4
ko    4
ar    4
Name: count, dtype: int64

Example 0 (te)
Question: ఒరెగాన్ రాష్ట్రంలోని అతిపెద్ద నగరం ఏది ?
Context: Portland is the largest city in the U.S. state of Oregon and the seat of Multnomah County. It is a major port in the Willamette Valley region of the Pacific Northwest, at the confluence of the Willamette and Columbia rivers. As of 2017, Portland had an estimated population of 647,805, making it the 26th-largest city in the United States, and the second-most populous in the Pacific Northwest (after Seattle). Approximately 2.4 million people live in the Portland metropolitan statistical area (MSA), making it the 25th most populous MSA in the United States. Its Combined Statistical Area (CSA) ranks 18th-largest with a population of around 3.2 million. Approximately 60% of Oregon's population resides within the Portland metropolitan area.
Gold answer: 'Portland'
Prediction: 'Seattle'

Example 2 (te)
Question: కలరా వ్యాధిని మొదటగా ఏ దేశంలో కన